# Exp 7 Orientation Pipeline — Synthetic Validation

Validates the gradient-based orientation estimator (Exp 7) against known ground-truth angles
using synthetic boulder images, analogous to `test_orientation_bug.ipynb` Parts 1–2
which validated the standard `fitEllipse` pipeline.

**Core comparison for each test:**
- **Standard pipeline**: pixel-aligned binary mask → `CHAIN_APPROX_NONE` polygon → `segmentize` → `fitEllipse` → `boulder_row`
- **Exp 7**: Canny on the raw synthetic image → `cv2.fitEllipse` on edge pixels within the mask

Expected result: Standard shows the 0°/90°/180° snap artifact; Exp 7 tracks ground-truth angle.

**Tests:**
1. Angle sweep (0°–165°, step 15°) — basic recovery
2. Size sweep — small / medium / large boulders
3. Aspect ratio sweep — mildly to strongly elongated
4. Noise robustness — Gaussian noise at increasing sigma
5. Edge contrast — low vs high boulder-to-background contrast
6. Summary comparison plot

In [ ]:
import sys
sys.path.insert(0, "/scratch/users/cayleigh/YOLOv8-BeyondEarth/src")

import typing_extensions
if not hasattr(typing_extensions, "TypeIs"):
    typing_extensions.TypeIs = typing_extensions.TypeGuard

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

from shapely.geometry import Polygon
from shapely import segmentize

from shptools_BOULDERING.geometry import fitEllipse
from shptools_BOULDERING.geomorph import boulder_row

RNG = np.random.default_rng(seed=42)
TEST_ANGLES = list(range(0, 180, 15))   # 0, 15, 30 … 165
print(f"Test angles: {TEST_ANGLES}")

In [ ]:
# ── Synthetic tile generator ───────────────────────────────────────────────
def make_synthetic_tile(a_px, b_px, theta_deg, img_size=128,
                         bg_value=100, fg_value=200, noise_sigma=0.0):
    """
    Draw a filled ellipse on a uniform gray background.

    Returns:
      tile_gray : (img_size, img_size) uint8 — grayscale image with the boulder
      mask_bool : (img_size, img_size) bool  — pixel-aligned rasterized mask

    The boulder edge in tile_gray is the step between bg_value and fg_value.
    Canny will detect this edge at the physical ellipse boundary.
    The mask is rasterized with cv2.ellipse — pixel-aligned, same as the pipeline.
    """
    cx = cy = img_size // 2
    # cv2.ellipse angle: CW from x-axis in image coords → same convention as theta_deg
    tile = np.full((img_size, img_size), bg_value, dtype=np.float32)
    cv2.ellipse(tile, (cx, cy), (a_px, b_px), float(theta_deg), 0, 360,
                float(fg_value), -1)  # filled ellipse

    if noise_sigma > 0:
        tile += RNG.normal(0, noise_sigma, tile.shape).astype(np.float32)
        tile = np.clip(tile, 0, 255)

    tile_gray = tile.astype(np.uint8)

    # Pixel-aligned binary mask — same rasterization as the real pipeline
    mask = np.zeros((img_size, img_size), dtype=np.uint8)
    cv2.ellipse(mask, (cx, cy), (a_px, b_px), float(theta_deg), 0, 360, 255, -1)
    mask_bool = mask > 0

    return tile_gray, mask_bool


# ── Exp 7 on numpy arrays ─────────────────────────────────────────────────
def orient_exp7(tile_gray, mask_bool):
    """
    Gradient-based orientation on numpy arrays (no rasterio needed for synthetic data).
    Returns (theta_deg, angle180, aspect_ratio) or None.
    """
    mask_u8 = mask_bool.astype(np.uint8)
    edges_full   = cv2.Canny(tile_gray, threshold1=15, threshold2=45)
    dilated_mask = cv2.dilate(mask_u8, np.ones((3, 3), np.uint8), iterations=2)
    edges        = cv2.bitwise_and(edges_full, edges_full, mask=dilated_mask)

    edge_pts = np.argwhere(edges)
    if len(edge_pts) < 5:
        return None

    edge_xy = edge_pts[:, ::-1].astype(np.float32).reshape(-1, 1, 2)
    try:
        (_, _), (MA, ma), angle_deg = cv2.fitEllipse(edge_xy)
    except cv2.error:
        return None

    if ma <= 0:
        return None

    aspect_ratio = max(MA, ma) / min(MA, ma)
    theta_deg    = (90.0 - angle_deg) % 180
    angle180     = angle_deg % 180
    return theta_deg, angle180, aspect_ratio


# ── Standard pipeline on pixel-aligned mask ───────────────────────────────
def orient_standard(mask_bool, res=1.0):
    """
    Standard mask → polygon → segmentize → fitEllipse → boulder_row.
    Returns (theta_deg, angle180, aspect_ratio) or None.
    """
    mask_u8 = mask_bool.astype(np.uint8) * 255
    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return None
    pts = max(contours, key=cv2.contourArea).squeeze(1)
    if len(pts) < 4:
        return None
    poly    = Polygon(pts.astype(float))
    row_seg = pd.Series({"geometry": segmentize(poly, res)})
    try:
        ellipse_poly, a_fit, b_fit, theta_rad = fitEllipse(row_seg)
    except Exception:
        return None
    theta_deg = np.degrees(theta_rad)
    try:
        mrr_row = pd.Series({"geometry": ellipse_poly.minimum_rotated_rectangle})
        vals    = boulder_row(mrr_row)
        long_axis, short_axis, angle180 = vals[2], vals[3], vals[7]
        aspect_ratio = long_axis / short_axis if short_axis > 0 else None
    except Exception:
        angle180, aspect_ratio = None, None
    return theta_deg, angle180, aspect_ratio


def wrap_to_ref(v, ref):
    """Wrap v to the equivalent angle (mod 180) closest to ref."""
    k = round((ref - v) / 180)
    return v + k * 180


print("Helpers defined.")

## Test 1: Angle sweep

Medium-sized boulder (a=30px, b=20px, 128×128 tile) at every 15° from 0° to 165°.
Both pipelines run on the same synthetic tile — same image, same mask.
Expected: Exp 7 tracks the diagonal; Standard snaps to 0°/90°/180°.

In [ ]:
A, B, IMG = 30, 20, 128

t1_records = []
for theta_true in TEST_ANGLES:
    tile_gray, mask_bool = make_synthetic_tile(A, B, theta_true, img_size=IMG)

    r_exp7 = orient_exp7(tile_gray, mask_bool)
    r_std  = orient_standard(mask_bool)

    exp7_theta  = wrap_to_ref(r_exp7[0], theta_true) if r_exp7 else None
    std_theta   = wrap_to_ref(r_std[0],  theta_true) if r_std  else None
    exp7_a180   = r_exp7[1] if r_exp7 else None
    std_a180    = r_std[1]  if r_std  else None

    t1_records.append({
        "theta_true":  theta_true,
        "exp7_theta":  exp7_theta,
        "std_theta":   std_theta,
        "exp7_angle180": exp7_a180,
        "std_angle180":  std_a180,
    })

df_t1 = pd.DataFrame(t1_records)
print(df_t1[["theta_true", "exp7_theta", "std_theta"]].to_string(index=False))

exp7_err = np.abs(df_t1["exp7_theta"] - df_t1["theta_true"]).dropna()
std_err  = np.abs(df_t1["std_theta"]  - df_t1["theta_true"]).dropna()
print(f"\nExp 7  MAE: {exp7_err.mean():.2f}°  max: {exp7_err.max():.2f}°")
print(f"Standard MAE: {std_err.mean():.2f}°  max: {std_err.max():.2f}°")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, col, label, color in [
    (axes[0], "exp7_theta", "Exp 7 (image gradient)",   "darkorange"),
    (axes[1], "std_theta",  "Standard (mask polygon)",  "steelblue"),
]:
    ax.plot([0, 165], [0, 165], 'k--', lw=1.5, label="perfect recovery")
    ax.scatter(df_t1["theta_true"], df_t1[col], color=color, s=60, zorder=3, label=label)
    ax.set_xlabel("Input angle (°)")
    ax.set_ylabel("Recovered angle (°)")
    ax.set_xlim(-5, 170); ax.set_ylim(-5, 185)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    err = (df_t1[col] - df_t1["theta_true"]).abs().mean()
    ax.set_title(f"{label}\nMAE = {err:.1f}°")

plt.suptitle(f"Test 1: Angle sweep  (a={A}px, b={B}px, {IMG}×{IMG} tile, no noise)\n"
             "Perfect recovery = points on the dashed diagonal", fontsize=11)
plt.tight_layout()
plt.savefig("exp7_val_test1_angle_sweep.png", dpi=150)
plt.show()

## Test 2: Size sweep

Fixed aspect ratio (a/b = 1.5), angle = 45°, three sizes.
Checks whether Exp 7 degrades for small boulders (few edge pixels).

In [ ]:
SIZES = [
    (8,  5,  64,  "small  (a=8px)"),
    (15, 10, 64,  "medium (a=15px)"),
    (30, 20, 128, "large  (a=30px)"),
    (50, 33, 200, "xlarge (a=50px)"),
]

t2_records = []
for a_px, b_px, img_size, size_label in SIZES:
    for theta_true in TEST_ANGLES:
        tile_gray, mask_bool = make_synthetic_tile(a_px, b_px, theta_true, img_size=img_size)
        r_exp7 = orient_exp7(tile_gray, mask_bool)
        r_std  = orient_standard(mask_bool)
        t2_records.append({
            "size_label": size_label, "a_px": a_px,
            "theta_true": theta_true,
            "exp7_theta": wrap_to_ref(r_exp7[0], theta_true) if r_exp7 else None,
            "std_theta":  wrap_to_ref(r_std[0],  theta_true) if r_std  else None,
        })

df_t2 = pd.DataFrame(t2_records)

fig, axes = plt.subplots(2, len(SIZES), figsize=(5 * len(SIZES), 10))
for col, (a_px, b_px, img_size, size_label) in enumerate(SIZES):
    sub = df_t2[df_t2["size_label"] == size_label]

    for row_i, (method_col, color, method_label) in enumerate([
        ("exp7_theta", "darkorange", "Exp 7"),
        ("std_theta",  "steelblue",  "Standard"),
    ]):
        ax = axes[row_i, col]
        ax.plot([0, 165], [0, 165], 'k--', lw=1.5)
        ax.scatter(sub["theta_true"], sub[method_col], color=color, s=50, zorder=3)
        err = (sub[method_col] - sub["theta_true"]).abs().mean()
        ax.set_title(f"{size_label}\n{method_label}  MAE={err:.1f}°", fontsize=9)
        ax.set_xlabel("Input angle (°)")
        ax.set_ylabel("Recovered angle (°)")
        ax.set_xlim(-5, 170); ax.set_ylim(-5, 185)
        ax.grid(True, alpha=0.3)

plt.suptitle("Test 2: Size sweep  (a/b=1.5, 45° angle, no noise)\n"
             "Row 1 = Exp 7 (gradient), Row 2 = Standard (mask polygon)", fontsize=11)
plt.tight_layout()
plt.savefig("exp7_val_test2_size_sweep.png", dpi=150)
plt.show()

print(f"\n{'Size':<20}  {'Exp7 MAE':>9}  {'Std MAE':>9}")
print("-" * 42)
for _, size_label in [(r, s) for r, _, _, s in SIZES]:
    sub = df_t2[df_t2["size_label"] == size_label]
    e7  = (sub["exp7_theta"] - sub["theta_true"]).abs().mean()
    std = (sub["std_theta"]  - sub["theta_true"]).abs().mean()
    print(f"{size_label:<20}  {e7:>9.2f}°  {std:>9.2f}°")

## Test 3: Aspect ratio sweep

Fixed size (a=30px, 128×128), angle = 45°, varying aspect ratio from 1.1 to 3.0.
Very circular boulders (AR≈1) have no dominant edge direction — Exp 7 may be
unreliable there (which is fine, since orientation is physically meaningless for circles).

In [ ]:
ASPECT_RATIOS = [1.05, 1.1, 1.2, 1.3, 1.5, 1.75, 2.0, 2.5, 3.0]
A_FIXED = 30

t3_records = []
for ar in ASPECT_RATIOS:
    b_px = max(1, int(A_FIXED / ar))
    for theta_true in TEST_ANGLES:
        tile_gray, mask_bool = make_synthetic_tile(A_FIXED, b_px, theta_true, img_size=128)
        r_exp7 = orient_exp7(tile_gray, mask_bool)
        t3_records.append({
            "ar": ar, "b_px": b_px, "theta_true": theta_true,
            "exp7_theta": wrap_to_ref(r_exp7[0], theta_true) if r_exp7 else None,
            "exp7_ar":    r_exp7[2] if r_exp7 else None,
        })

df_t3 = pd.DataFrame(t3_records)

# MAE per aspect ratio
mae_by_ar = (df_t3.groupby("ar")
             .apply(lambda g: (g["exp7_theta"] - g["theta_true"]).abs().mean())
             .reset_index(name="mae"))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(mae_by_ar["ar"], mae_by_ar["mae"], 'o-', color="darkorange")
axes[0].axvline(1.2, color='k', linestyle='--', label="AR=1.2 (filter threshold)")
axes[0].axhline(10,  color='r', linestyle=':', alpha=0.5, label="10° error threshold")
axes[0].set_xlabel("True aspect ratio (a/b)")
axes[0].set_ylabel("Mean absolute error (°)")
axes[0].set_title("Exp 7 angle MAE vs aspect ratio")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Scatter at AR=1.5 vs AR=1.1 to show qualitative difference
for ar_val, color in [(1.1, 'gray'), (1.5, 'darkorange'), (2.0, 'darkred')]:
    sub = df_t3[df_t3["ar"] == ar_val]
    axes[1].scatter(sub["theta_true"], sub["exp7_theta"],
                    label=f"AR={ar_val}", alpha=0.8, s=40, color=color)
axes[1].plot([0, 165], [0, 165], 'k--', lw=1.5)
axes[1].set_xlabel("Input angle (°)"); axes[1].set_ylabel("Recovered angle (°)")
axes[1].set_title("Angle recovery at AR=1.1, 1.5, 2.0")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle("Test 3: Aspect ratio sweep  (a=30px, θ=0–165°, no noise)", fontsize=11)
plt.tight_layout()
plt.savefig("exp7_val_test3_ar_sweep.png", dpi=150)
plt.show()

## Test 4: Noise robustness

Medium boulder (a=30px, b=20px), angle=45°, increasing Gaussian noise sigma.
Background=100, foreground=200, so edge contrast = 100 grey levels.
Tests SNR at which Exp 7 starts failing.

In [ ]:
NOISE_SIGMAS = [0, 5, 10, 20, 30, 50, 75, 100]
N_TRIALS     = 20   # repeat each sigma with different RNG seeds

t4_records = []
for sigma in NOISE_SIGMAS:
    for trial in range(N_TRIALS):
        for theta_true in [15, 45, 75, 120]:   # representative subset of angles
            rng_local = np.random.default_rng(seed=trial * 1000 + int(sigma))
            tile_gray, mask_bool = make_synthetic_tile(
                30, 20, theta_true, img_size=128,
                bg_value=100, fg_value=200, noise_sigma=sigma)
            r_exp7 = orient_exp7(tile_gray, mask_bool)
            t4_records.append({
                "sigma":      sigma,
                "theta_true": theta_true,
                "exp7_theta": wrap_to_ref(r_exp7[0], theta_true) if r_exp7 else None,
                "failed":     r_exp7 is None,
            })

df_t4 = pd.DataFrame(t4_records)

# MAE and failure rate per sigma
t4_summary = (df_t4.groupby("sigma")
              .agg(
                  mae        = ("exp7_theta", lambda x: (x - df_t4.loc[x.index, "theta_true"]).abs().mean()),
                  fail_rate  = ("failed",     "mean"),
              ).reset_index())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(t4_summary["sigma"], t4_summary["mae"], 'o-', color="darkorange")
axes[0].axhline(10, color='r', linestyle=':', alpha=0.6, label="10° threshold")
axes[0].set_xlabel("Noise sigma (grey levels)"); axes[0].set_ylabel("MAE (°)")
axes[0].set_title("Exp 7 angle MAE vs image noise\n(edge contrast = 100 grey levels)")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(t4_summary["sigma"], t4_summary["fail_rate"] * 100, 's-', color="steelblue")
axes[1].set_xlabel("Noise sigma"); axes[1].set_ylabel("Failure rate (%)")
axes[1].set_title("Exp 7 failure rate vs noise\n(failure = fewer than 5 Canny edge pts)")
axes[1].set_ylim(-2, 105); axes[1].grid(True, alpha=0.3)

plt.suptitle("Test 4: Noise robustness  (a=30px, b=20px, 128×128)", fontsize=11)
plt.tight_layout()
plt.savefig("exp7_val_test4_noise.png", dpi=150)
plt.show()

print(t4_summary.to_string(index=False))

## Test 5: Edge contrast sweep

Fixed noise (sigma=20), varying boulder-to-background contrast.
HiRISE boulders vary significantly in contrast depending on lighting geometry and albedo.
Determines the minimum contrast at which Exp 7 is reliable.

In [ ]:
CONTRASTS    = [10, 20, 30, 50, 75, 100, 150]   # fg - bg in grey levels
NOISE_FIXED  = 20

t5_records = []
for contrast in CONTRASTS:
    for trial in range(N_TRIALS):
        for theta_true in [15, 45, 75, 120]:
            tile_gray, mask_bool = make_synthetic_tile(
                30, 20, theta_true, img_size=128,
                bg_value=100, fg_value=100 + contrast,
                noise_sigma=NOISE_FIXED)
            r_exp7 = orient_exp7(tile_gray, mask_bool)
            t5_records.append({
                "contrast":   contrast,
                "snr":        contrast / NOISE_FIXED,
                "theta_true": theta_true,
                "exp7_theta": wrap_to_ref(r_exp7[0], theta_true) if r_exp7 else None,
                "failed":     r_exp7 is None,
            })

df_t5 = pd.DataFrame(t5_records)

t5_summary = (df_t5.groupby(["contrast", "snr"])
              .agg(
                  mae       = ("exp7_theta", lambda x: (x - df_t5.loc[x.index, "theta_true"]).abs().mean()),
                  fail_rate = ("failed",     "mean"),
              ).reset_index())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(t5_summary["snr"], t5_summary["mae"], 'o-', color="darkorange")
axes[0].axhline(10, color='r', linestyle=':', alpha=0.6, label="10° threshold")
axes[0].set_xlabel("SNR (contrast / noise sigma)")
axes[0].set_ylabel("MAE (°)")
axes[0].set_title(f"Exp 7 MAE vs SNR  (noise sigma={NOISE_FIXED})")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(t5_summary["snr"], t5_summary["fail_rate"] * 100, 's-', color="steelblue")
axes[1].set_xlabel("SNR")
axes[1].set_ylabel("Failure rate (%)")
axes[1].set_title("Exp 7 failure rate vs SNR")
axes[1].set_ylim(-2, 105); axes[1].grid(True, alpha=0.3)

plt.suptitle("Test 5: Edge contrast sweep  (a=30px, b=20px, noise σ=20)", fontsize=11)
plt.tight_layout()
plt.savefig("exp7_val_test5_contrast.png", dpi=150)
plt.show()

print(t5_summary.to_string(index=False))

## Test 6: Head-to-head — Exp 7 vs Standard at all angles

The definitive comparison: same synthetic tile, same mask, two orientation estimators.
Run at multiple sizes to show the standard pipeline fails at ALL sizes while
Exp 7 tracks the diagonal.

In [ ]:
HEAD2HEAD_SIZES = [
    (10,  7,  64,  "small"),
    (20, 13, 100,  "medium"),
    (40, 27, 200,  "large"),
]

t6_records = []
for a_px, b_px, img_size, size_label in HEAD2HEAD_SIZES:
    for theta_true in TEST_ANGLES:
        tile_gray, mask_bool = make_synthetic_tile(a_px, b_px, theta_true, img_size=img_size)
        r_exp7 = orient_exp7(tile_gray, mask_bool)
        r_std  = orient_standard(mask_bool)
        t6_records.append({
            "size":       size_label,
            "theta_true": theta_true,
            "exp7":  wrap_to_ref(r_exp7[0], theta_true) if r_exp7 else None,
            "std":   wrap_to_ref(r_std[0],  theta_true) if r_std  else None,
        })

df_t6 = pd.DataFrame(t6_records)

fig, axes = plt.subplots(2, len(HEAD2HEAD_SIZES), figsize=(5.5 * len(HEAD2HEAD_SIZES), 10))

for col, (a_px, b_px, img_size, size_label) in enumerate(HEAD2HEAD_SIZES):
    sub = df_t6[df_t6["size"] == size_label]
    for row_i, (method, color, title) in enumerate([
        ("exp7", "darkorange", "Exp 7 (image gradient)"),
        ("std",  "steelblue",  "Standard (mask polygon)"),
    ]):
        ax = axes[row_i, col]
        ax.plot([0, 165], [0, 165], 'k--', lw=1.5, label="perfect recovery")
        ax.scatter(sub["theta_true"], sub[method], color=color, s=60, zorder=3)
        err = (sub[method] - sub["theta_true"]).abs().mean()
        ax.set_title(f"{size_label} — {title}\nMAE = {err:.1f}°", fontsize=9)
        ax.set_xlabel("Input angle (°)"); ax.set_ylabel("Recovered (°)")
        ax.set_xlim(-5, 170); ax.set_ylim(-5, 185)
        ax.grid(True, alpha=0.3)

plt.suptitle(
    "Test 6: Exp 7 vs Standard — head-to-head across sizes\n"
    "Row 1 (orange) = Exp 7   |   Row 2 (blue) = Standard mask polygon\n"
    "Points on the dashed diagonal = correct recovery",
    fontsize=11)
plt.tight_layout()
plt.savefig("exp7_val_test6_head2head.png", dpi=150)
plt.show()

print(f"\n{'Size':<8}  {'Exp7 MAE':>9}  {'Std MAE':>9}")
print("-" * 32)
for _, _, _, size_label in HEAD2HEAD_SIZES:
    sub = df_t6[df_t6["size"] == size_label]
    e7  = (sub["exp7"] - sub["theta_true"]).abs().mean()
    std = (sub["std"]  - sub["theta_true"]).abs().mean()
    print(f"{size_label:<8}  {e7:>9.2f}°  {std:>9.2f}°")

## Summary

Consolidated view of Exp 7 performance across all tests.
Key metrics for the paper:
- Angle recovery MAE vs input angle (Test 1)
- Minimum SNR for reliable operation (Tests 4–5)
- Minimum boulder size (Test 2)
- Head-to-head vs standard pipeline (Test 6)

## Test 7: YOLO mask vs SAM2 mask as spatial filter

In Exp 7 the mask is used **only as a spatial filter** — to restrict which Canny edge
pixels are considered. The actual orientation comes from the image gradients, not the mask shape.

This test asks: does it matter whether the spatial filter comes from a YOLO bbox mask
or a SAM2 segmentation mask?

Three mask types are simulated on the same synthetic image:
- **Exact mask** — the true rasterized ellipse (best possible filter)
- **YOLO-like mask** — axis-aligned bounding rectangle of the boulder (coarse, always larger than the boulder, includes corners)
- **SAM2-like mask** — slightly dilated ellipse (closer to the true boulder shape, small overreach)

If all three give the same orientation → the mask quality doesn't matter for Exp 7,
and YOLO masks are sufficient as spatial filters.
If YOLO-like diverges → the rectangular corners capture spurious background edges
that pull the fitted ellipse away from the true orientation.

In [ ]:
def make_yolo_like_mask(mask_bool):
    """
    Simulate a YOLO bbox mask: axis-aligned bounding rectangle of the true mask.
    This is the coarsest possible spatial filter — always rectangular, always
    larger than the boulder, includes background corners.
    """
    rows_hit = np.any(mask_bool, axis=1)
    cols_hit = np.any(mask_bool, axis=0)
    if not rows_hit.any() or not cols_hit.any():
        return mask_bool.copy()
    r0, r1 = np.where(rows_hit)[0][[0, -1]]
    c0, c1 = np.where(cols_hit)[0][[0, -1]]
    bbox_mask = np.zeros_like(mask_bool)
    bbox_mask[r0:r1 + 1, c0:c1 + 1] = True
    return bbox_mask


def make_sam2_like_mask(mask_bool, dilation_px=3):
    """
    Simulate a SAM2-like mask: small dilation of the true ellipse mask.
    SAM2 generally segments the boulder accurately but overshoots the boundary
    by a pixel or two. This is a tighter filter than the YOLO bbox.
    """
    mask_u8  = mask_bool.astype(np.uint8)
    dilated  = cv2.dilate(mask_u8, np.ones((dilation_px, dilation_px), np.uint8))
    return dilated > 0


# ── Run Test 7 ────────────────────────────────────────────────────────────
MASK_TYPES = [
    ("exact_mask",    lambda m: m,                    "Exact (true ellipse)", "black"),
    ("sam2_like",     make_sam2_like_mask,             "SAM2-like (dilated)",  "tomato"),
    ("yolo_like",     make_yolo_like_mask,             "YOLO-like (bbox rect)","steelblue"),
]

# Test at several sizes and aspect ratios to be thorough
T7_CONFIGS = [
    (15, 10,  64, "small  a=15"),
    (30, 20, 128, "medium a=30"),
    (50, 33, 200, "large  a=50"),
]

t7_records = []
for a_px, b_px, img_size, cfg_label in T7_CONFIGS:
    for theta_true in TEST_ANGLES:
        tile_gray, exact_mask = make_synthetic_tile(
            a_px, b_px, theta_true, img_size=img_size,
            bg_value=100, fg_value=200, noise_sigma=10)

        for mask_key, mask_fn, _, _ in MASK_TYPES:
            spatial_mask = mask_fn(exact_mask)
            result = orient_exp7(tile_gray, spatial_mask)
            t7_records.append({
                "config":     cfg_label,
                "mask_type":  mask_key,
                "theta_true": theta_true,
                "exp7_theta": wrap_to_ref(result[0], theta_true) if result else None,
                "failed":     result is None,
            })

df_t7 = pd.DataFrame(t7_records)

# ── Summary table ─────────────────────────────────────────────────────────
print(f"{'Config':<14}  {'Mask type':<25}  {'MAE (°)':>8}  {'Fail%':>6}")
print("-" * 60)
for cfg_label, _, _, _ in T7_CONFIGS:
    for mask_key, _, mask_label, _ in MASK_TYPES:
        sub = df_t7[(df_t7["config"] == cfg_label) & (df_t7["mask_type"] == mask_key)]
        mae  = (sub["exp7_theta"] - sub["theta_true"]).abs().mean()
        fail = sub["failed"].mean() * 100
        print(f"{cfg_label:<14}  {mask_label:<25}  {mae:>8.2f}°  {fail:>5.1f}%")
    print()

In [ ]:
# ── Test 7 plots ──────────────────────────────────────────────────────────
n_configs = len(T7_CONFIGS)
fig, axes = plt.subplots(n_configs, 3, figsize=(15, 5 * n_configs))

for row_i, (a_px, b_px, img_size, cfg_label) in enumerate(T7_CONFIGS):
    for col_i, (mask_key, _, mask_label, color) in enumerate(MASK_TYPES):
        ax  = axes[row_i, col_i]
        sub = df_t7[(df_t7["config"] == cfg_label) & (df_t7["mask_type"] == mask_key)]
        mae = (sub["exp7_theta"] - sub["theta_true"]).abs().mean()

        ax.plot([0, 165], [0, 165], 'k--', lw=1.5)
        ax.scatter(sub["theta_true"], sub["exp7_theta"],
                   color=color, s=55, zorder=3)
        ax.set_title(f"{cfg_label}\n{mask_label}\nMAE = {mae:.1f}°", fontsize=9)
        ax.set_xlabel("Input angle (°)")
        ax.set_ylabel("Recovered (°)")
        ax.set_xlim(-5, 170); ax.set_ylim(-5, 185)
        ax.grid(True, alpha=0.3)

plt.suptitle(
    "Test 7: YOLO-like bbox mask vs SAM2-like mask vs exact mask — Exp 7 spatial filter comparison\n"
    "If all three track the diagonal equally → mask quality doesn't matter for gradient orientation\n"
    "If YOLO-like (blue) diverges → rectangular corners introduce spurious background edges",
    fontsize=11)
plt.tight_layout()
plt.savefig("exp7_val_test7_mask_comparison.png", dpi=150)
plt.show()

# ── Visualise the three mask types for one example ─────────────────────────
theta_example = 45
tile_ex, exact_ex = make_synthetic_tile(30, 20, theta_example, img_size=128,
                                         bg_value=100, fg_value=200, noise_sigma=10)
sam2_ex  = make_sam2_like_mask(exact_ex)
yolo_ex  = make_yolo_like_mask(exact_ex)

fig2, axes2 = plt.subplots(2, 4, figsize=(18, 9))

# Top row: masks overlaid on the tile
for ax, mask, title in zip(
    axes2[0],
    [exact_ex, sam2_ex, yolo_ex],
    ["Exact mask", "SAM2-like mask\n(dilated)", "YOLO-like mask\n(bbox rect)"],
):
    overlay = np.stack([tile_ex] * 3, axis=-1)
    overlay[mask, 0] = 255   # mask region tinted red
    ax.imshow(overlay)
    ax.set_title(title, fontsize=10)
    ax.axis("off")

axes2[0, 3].axis("off")

# Bottom row: Canny edges captured by each mask
dilated_exact = cv2.dilate(exact_ex.astype(np.uint8), np.ones((3,3), np.uint8), iterations=2)
dilated_sam2  = cv2.dilate(sam2_ex.astype(np.uint8),  np.ones((3,3), np.uint8), iterations=2)
dilated_yolo  = cv2.dilate(yolo_ex.astype(np.uint8),  np.ones((3,3), np.uint8), iterations=2)
edges_full_ex = cv2.Canny(tile_ex, 15, 45)

for ax, dmask, title in zip(
    axes2[1],
    [dilated_exact, dilated_sam2, dilated_yolo],
    ["Edges (exact)", "Edges (SAM2-like)", "Edges (YOLO-like)"],
):
    edges = cv2.bitwise_and(edges_full_ex, edges_full_ex, mask=dmask)
    ax.imshow(edges, cmap='hot')
    ax.set_title(title, fontsize=10)
    ax.axis("off")

axes2[1, 3].axis("off")

plt.suptitle(f"Mask types and captured Canny edges  (θ={theta_example}°, a=30px, b=20px)\n"
             "Top: mask overlay on tile    Bottom: Canny edges within dilated mask",
             fontsize=11)
plt.tight_layout()
plt.savefig("exp7_val_test7_mask_visualisation.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: angle recovery (Test 1, medium boulder, no noise)
axes[0].plot([0, 165], [0, 165], 'k--', lw=1.5, label="perfect")
axes[0].scatter(df_t1["theta_true"], df_t1["exp7_theta"],
                color="darkorange", s=70, zorder=3, label="Exp 7")
axes[0].scatter(df_t1["theta_true"], df_t1["std_theta"],
                color="steelblue",  s=40, zorder=2, alpha=0.7, label="Standard", marker='s')
axes[0].set_xlabel("Input angle (°)"); axes[0].set_ylabel("Recovered (°)")
axes[0].set_title(f"Angle recovery\n(a=30px, b=20px, no noise)")
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

# Panel 2: MAE vs noise sigma (Test 4)
axes[1].plot(t4_summary["sigma"], t4_summary["mae"],
             'o-', color="darkorange", label="Exp 7 MAE")
axes[1].axhline(10, color='r', linestyle=':', alpha=0.7, label="10° threshold")
axes[1].set_xlabel("Noise sigma (grey levels)")
axes[1].set_ylabel("MAE (°)")
axes[1].set_title("Noise robustness\n(edge contrast = 100)")
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

# Panel 3: MAE vs aspect ratio (Test 3)
axes[2].plot(mae_by_ar["ar"], mae_by_ar["mae"],
             'o-', color="darkorange", label="Exp 7 MAE")
axes[2].axvline(1.2, color='k', linestyle='--', alpha=0.7, label="AR filter threshold")
axes[2].axhline(10,  color='r', linestyle=':', alpha=0.7, label="10° threshold")
axes[2].set_xlabel("True aspect ratio (a/b)")
axes[2].set_ylabel("MAE (°)")
axes[2].set_title("Aspect ratio dependency\n(a=30px, no noise)")
axes[2].legend(fontsize=9); axes[2].grid(True, alpha=0.3)

plt.suptitle("Exp 7 synthetic validation — summary", fontsize=12)
plt.tight_layout()
plt.savefig("exp7_val_summary.png", dpi=150)
plt.show()